In [1]:
using PyPlot
using LinearAlgebra
using SparseArrays
using Statistics

# The linear and nonlinear (elastic) wave equation

The linear wave equation:
$$
\begin{cases}
  \partial_t\,u \,+\,\partial_x\,v\,&=\,0,
  \\[0.3em]
  \partial_t\,v \,+c^2\,\partial_x\,u\,&=\,0.
\end{cases}
$$

The nonlinear elasticity-wave equation:
$$
\begin{cases}
  \partial_t\,u \,+\,\partial_x\,v\,&=\,0,
  \\[0.3em]
  \partial_t\,v \,+c^2\,(1+\partial_x\,u)(1+\tfrac12\partial_x\,u)\partial_x\,u\,&=\,0.
\end{cases}
$$

## Boundary conditions:

Dirichlet conditions on $u$ correspond to a fixed displacement, i.e., the string/beam is attached to something.
$$
\text{left boundary:}\quad
u(t,a) = 0,
\qquad\qquad
\text{right boundary:}\quad
u(t,b) = 0.
$$

Neumann conditions on $u$ correspond to "stress free boundary conditions" (if we set the normal derivative to zero), or a "fixed stress" boundary condition that models a weight pulling on the string/beam:
$$
\text{left boundary:}\quad
\partial_x u(t,a) = g(a),
\qquad\qquad
\text{right boundary:}\quad
\partial_x u(t,b) = g(b).
$$

## Forcing terms:

A force $f(t,x)$ acts on the momentum equation, which reads (in second order form):
$$
\partial_t^2u + \partial_x^2u \;=\; f(t,x)
$$
We can translate this into first order form by finding a function $F(t, x)$ with $\partial_xF(t,x) = f(t,x)$:

The linear wave equation:
$$
\begin{cases}
  \partial_t\,u \,+\,\partial_x\,v\,&=\,0,
  \\[0.3em]
  \partial_t\,v \,+c^2\,\partial_x\,u\,&=\,-F(t,x).
\end{cases}
$$

The nonlinear elasticity-wave equation:
$$
\begin{cases}
  \partial_t\,u \,+\,\partial_x\,v\,&=\,0,
  \\[0.3em]
  \partial_t\,v \,+c^2\,(1+\partial_x\,u)(1+\tfrac12\partial_x\,u)\partial_x\,u\,&=\,-F(t,x).
\end{cases}
$$

In [10]:
# A string with a weight distribution x^κ * (κ + 1):

c = 1.00
stab = 16.0

@inline function analytic_solution(t, x)
    u = 0
    v = 0
    return u, v
end

# For domain [0, 1]
gravity = 0.1
κ = 0
@inline function F(t, x)
    return 0., gravity * (x^(κ + 1.) - 1. / (κ + 2.))
end

@inline function g(t, x)
    return 0., 0.
end;

In [3]:
# A string with a point weight at the end:

c = 1.00
stab = 16.0

@inline function analytic_solution(t, x)
    u = 0
    v = 0
    return u, v
end

gravity = 0.1

@inline function F(t, x)
    return 0., 0.
end

@inline function g(t, x)
    return gravity, 0.
end ;

# Finite difference scheme with stabilization:

In [4]:
@enum BoundaryType do_nothing dirichlet_u dirichlet_v neumann_u neumann_v periodic

struct Discretization
    xs::Vector{Float64}
    left_boundary::BoundaryType
    right_boundary::BoundaryType
    filter_average_v::Bool
end

In [5]:
# Captures c, analytic_solution, F, g

function euler_step_linear(discretization, CFL, stabilization, U_old, t_old)
    h = discretization.xs[2] - discretization.xs[1]

    u_old = U_old[1,:]
    v_old = U_old[2,:]

    τ = CFL / c * h

    u_new = zeros(2^k + 2)
    v_new = zeros(2^k + 2)

    #
    # Central difference for interior degrees of freedom:
    #
    
    for i in 2 : 2^k + 1
        F_i = F(t, discretization.xs[i])
        
        u_new[i] = u_old[i] - τ / (2.0 * h) * (v_old[i+1] - v_old[i-1]) - τ * F_i[1]
        u_prime = 1.0 / (2.0 * h) * (u_old[i+1] - u_old[i-1])
        v_new[i] = v_old[i] - c^2 * τ * u_prime - τ * F_i[2]

        u_new[i] = u_new[i] + stabilization * τ * (u_old[i+1] - 2.0 * u_old[i] + u_old[i-1])
        v_new[i] = v_new[i] + stabilization * τ * (v_old[i+1] - 2.0 * v_old[i] + v_old[i-1])
    end

    #
    # One-sided difference for boundary degrees of freedom:
    #

    F_1 = F(t, discretization.xs[1])
    u_new[1] = u_old[1] - τ / (1.0 * h) * (v_old[2] - v_old[1]) - τ * F_1[1]
    u_prime = 1.0 / (1.0 * h) * (u_old[2] - u_old[1])
    v_new[1] = v_old[1] - c^2 * τ * u_prime - τ * F_1[2]

    N = 2^k + 2
    F_N = F(t, discretization.xs[N])
    u_new[N] = u_old[N] - τ / (1.0 * h) * (v_old[N] - v_old[N-1]) - τ * F_N[1]
    u_prime = 1.0 / (1.0 * h) * (u_old[N] - u_old[N-1])
    v_new[N] = v_old[N] - c^2 * τ * u_prime - τ * F_N[2]

    #
    # Boundary post-processing:
    #

    if discretization.left_boundary == periodic || discretization.right_boundary ==  periodic
        @assert(discretization.left_boundary == periodic && discretization.right_boundary == periodic)
        
        F_1 = F(t, discretization.xs[1])
        u_new[1] = u_old[1] - τ / (2.0 * h) * (v_old[2] - v_old[N-1]) - τ * F_1[1]
        u_prime = 1.0 / (2.0 * h) * (u_old[2] - u_old[N-1])
        v_new[1] = v_old[1] - c^2 * τ * u_prime - τ * F_1[2]
        u_new[1] = u_new[1] + stabilization * τ * (u_old[2] - 2.0 * u_old[1] + u_old[N-1])
        v_new[1] = v_new[1] + stabilization * τ * (v_old[2] - 2.0 * v_old[1] + v_old[N-1])
        u_new[N] = u_new[1]
        v_new[N] = v_old[1]
    end

    if discretization.left_boundary == dirichlet_u
        u_new[1], dummy = analytic_solution(t + τ, discretization.xs[1])        
    elseif discretization.left_boundary == dirichlet_v
        dummy, v_new[1] = analytic_solution(t + τ, discretization.xs[1])
    elseif discretization.left_boundary == neumann_u
        g_1 = g(t + τ, discretization.xs[1])
        u_new[1] = u_new[2] - h * g_1[1]
    elseif discretization.left_boundary == neumann_v
        @error "Not implemented"
    end

    if discretization.right_boundary == dirichlet_u
        u_new[N], dummy = analytic_solution(t + τ, discretization.xs[N])        
    elseif discretization.right_boundary == dirichlet_v
        dummy, v_new[N] = analytic_solution(t + τ, discretization.xs[N])
    elseif discretization.right_boundary == neumann_u
        g_N = g(t + τ, discretization.xs[N])
        u_new[N] = u_new[N-1] + h * g_N[1]
    elseif discretization.right_boundary == neumann_v
        @error "Not implemented"
    end
    
    if discretization.filter_average_v
        mean_v = mean(v_new)
        v_new .-= mean_v
    end
    
    return vcat(u_new', v_new'), τ
end

euler_step_linear (generic function with 1 method)

In [6]:
# Captures c, analytic_solution, F, g

function euler_step_nonlinear(discretization, CFL, stabilization, U_old, t_old) 
    h = discretization.xs[2] - discretization.xs[1]

    u_old = U_old[1,:]
    v_old = U_old[2,:]

    τ = CFL / c * h

    u_new = zeros(2^k + 2)
    v_new = zeros(2^k + 2)

    #
    # Central difference for interior degrees of freedom:
    #
    
    for i in 2 : 2^k + 1
        F_i = F(t, discretization.xs[i])
        
        u_new[i] = u_old[i] - τ / (2.0 * h) * (v_old[i+1] - v_old[i-1]) - τ * F_i[1]
        u_prime = 1.0 / (2.0 * h) * (u_old[i+1] - u_old[i-1])
        v_new[i] = v_old[i] - c^2 * τ * (1 + u_prime) * (1 + 0.5 * u_prime) * u_prime - τ * F_i[2]

        u_new[i] = u_new[i] + stabilization * τ * (u_old[i+1] - 2.0 * u_old[i] + u_old[i-1])
        v_new[i] = v_new[i] + stabilization * τ * (v_old[i+1] - 2.0 * v_old[i] + v_old[i-1])
    end

    #
    # One-sided difference for boundary degrees of freedom:
    #

    F_1 = F(t, discretization.xs[1])
    u_new[1] = u_old[1] - τ / (1.0 * h) * (v_old[2] - v_old[1]) - τ * F_1[1]
    u_prime = 1.0 / (1.0 * h) * (u_old[2] - u_old[1])
    v_new[1] = v_old[1] - c^2 * τ * (1 + u_prime) * (1 + 0.5 * u_prime) * u_prime - τ * F_1[2]

    N = 2^k + 2
    F_N = F(t, discretization.xs[N])
    u_new[N] = u_old[N] - τ / (1.0 * h) * (v_old[N] - v_old[N-1]) - τ * F_N[1]
    u_prime = 1.0 / (1.0 * h) * (u_old[N] - u_old[N-1])
    v_new[N] = v_old[N] - c^2 * τ * (1 + u_prime) * (1 + 0.5 * u_prime) * u_prime - τ * F_N[2]

    #
    # Boundary value post-processing:
    #

    if discretization.left_boundary == periodic || discretization.right_boundary ==  periodic
        @assert(discretization.left_boundary == periodic && discretization.right_boundary == periodic)
        
        F_1 = F(t, discretization.xs[1])
        u_new[1] = u_old[1] - τ / (2.0 * h) * (v_old[2] - v_old[N-1]) - τ * F_1[1]
        u_prime = 1.0 / (2.0 * h) * (u_old[2] - u_old[N-1])
        v_new[1] = v_old[1] - c^2 * τ * (1 + u_prime) * (1 + 0.5 * u_prime) * u_prime - τ * F_1[2]
        u_new[1] = u_new[1] + stabilization * τ * (u_old[2] - 2.0 * u_old[1] + u_old[N-1])
        v_new[1] = v_new[1] + stabilization * τ * (v_old[2] - 2.0 * v_old[1] + v_old[N-1])
        u_new[N] = u_new[1]
        v_new[N] = v_old[1]
    end
    
    if discretization.left_boundary == dirichlet_u
        u_new[1], dummy = analytic_solution(t + τ, discretization.xs[1])        
    elseif discretization.left_boundary == dirichlet_v
        dummy, v_new[1] = analytic_solution(t + τ, discretization.xs[1])
    elseif discretization.left_boundary == neumann_u
        g_1 = g(t + τ, discretization.xs[1])
        u_new[1] = u_new[2] - h * g_1[1]
    elseif discretization.left_boundary == neumann_v
        @error "Not implemented"
    end

    if discretization.right_boundary == dirichlet_u
        u_new[N], dummy = analytic_solution(t + τ, discretization.xs[N])        
    elseif discretization.right_boundary == dirichlet_v
        dummy, v_new[N] = analytic_solution(t + τ, discretization.xs[N])
    elseif discretization.right_boundary == neumann_u
        g_N = g(t + τ, discretization.xs[N])
        u_new[N] = u_new[N-1] + h * g_N[1]
    elseif discretization.right_boundary == neumann_v
        @error "Not implemented"
    end

    if discretization.filter_average_v
        mean_v = mean(v_new)
        v_new .-= mean_v
    end
    
    return vcat(u_new', v_new'), τ
end

euler_step_nonlinear (generic function with 1 method)

# The main loop (explicit Euler updates to final time)

In [7]:
# The domain Ω
Ω = [0.0, 1.0]
left_boundary = dirichlet_u
right_boundary = neumann_u

# Spatial discretization
k = 8
h = (Ω[2] - Ω[1]) / (2^k + 1)
xs = [Ω[1] + h * (i - 1) for i in 1:(2^k+2)]

filter_average_v = true

discretization = Discretization(xs, left_boundary, right_boundary, filter_average_v);

In [8]:
# Final time
T = 2.5
output_granularity = 0.1

CFL = 0.1
out = 1.0e-6

yrange = [0, 0.25] # for plotting

compute_error = true

plot_nonlinear = true
plot_linear = true
plot_v = false

;

In [12]:
@time begin
    U = zeros(2, 2^k + 2) # The two-component state vector [u, v]
    U[1,:] = [analytic_solution(0, x)[1] for x in xs]
    U[2,:] = [analytic_solution(0, x)[2] for x in xs]
    U_linear = copy(U)
    
    print("output at initial time t = 0\n")
    plt.figure(figsize=(10,7))
    if plot_nonlinear
        plot(xs, U[1,:])
    end
    if plot_linear
        plot(xs, U_linear[1,:])
    end
    if plot_v
        if plot_nonlinear
            plot(xs, U[2,:])
        end
        if plot_linear
            plot(xs, U_linear[2,:])
        end
    end
    if compute_error
        plot(xs, [analytic_solution(0, x)[1] for x in xs ])
    end
    plt.xlim(Ω)
    plt.ylim(yrange)
    plt.savefig("output-" * lpad(0,4,"0") * ".png")
    plt.close()
    output_cycle = 1
    
    t = 0
    cycle = 0
    while t < T
        U, τ = euler_step_nonlinear(discretization, CFL, stab, U, t)
        U_linear, τ_linear = euler_step_linear(discretization, CFL, stab, U_linear, t)
        
        @assert(abs(τ - τ_linear) < 1.e-10)
        if(τ < out * T)
            print("We didn't make it.\n")
            break
        end
            
        t = t + τ
        cycle = cycle + 1
        
        if (output_cycle * output_granularity < t)
            print("output at time t = ", t, " (τ = ", τ, ", cycle = ", cycle, ")\n")
            plt.figure(figsize=(10,7))
            if plot_nonlinear
                plot(xs, U[1,:])
            end
            if plot_linear
                plot(xs, U_linear[1,:])
            end
            if plot_v
                if plot_nonlinear
                    plot(xs, U[2,:])
                end
                if plot_linear
                    plot(xs, U_linear[2,:])
                end
            end
            if compute_error
                plot(xs, [analytic_solution(0, x)[1] for x in xs ])
            end
            plt.xlim(Ω)
            plt.ylim(yrange)
            plt.savefig("output-" * lpad(output_cycle,4,"0") * ".png")
            plt.close()
            output_cycle = output_cycle + 1
        end
    end;

    if compute_error
        # Compute error at final time:
        U_error = zeros(2, 2^k + 2) # The two-component state vector [u, v]
        U_error[1,:] = [analytic_solution(t, x)[1] for x in xs]
        U_error[2,:] = [analytic_solution(t, x)[2] for x in xs]
        U_error = U_error - U
        error_nonlinear = norm(U_error[1,:], Inf) #+ norm(U_error[2,:], Inf)
        println("L^∞ error nonlinear: ", error_nonlinear)
        
        U_error = zeros(2, 2^k + 2) # The two-component state vector [u, v]
        U_error[1,:] = [analytic_solution(t, x)[1] for x in xs]
        U_error[2,:] = [analytic_solution(t, x)[2] for x in xs]
        U_error = U_error - U_linear
        error_linear = norm(U_error[1,:], Inf) #+ norm(U_error[2,:], Inf)
        println("L^∞ error linear:    ", error_linear)
    end
end

output at initial time t = 0
output at time t = 0.10038910505836547 (τ = 0.0003891050583657588, cycle = 258)
output at time t = 0.20038910505836513 (τ = 0.0003891050583657588, cycle = 515)
output at time t = 0.3003891050583648 (τ = 0.0003891050583657588, cycle = 772)
output at time t = 0.40038910505836445 (τ = 0.0003891050583657588, cycle = 1029)
output at time t = 0.5003891050583641 (τ = 0.0003891050583657588, cycle = 1286)
output at time t = 0.6003891050583637 (τ = 0.0003891050583657588, cycle = 1543)
output at time t = 0.7003891050583634 (τ = 0.0003891050583657588, cycle = 1800)
output at time t = 0.800389105058363 (τ = 0.0003891050583657588, cycle = 2057)
output at time t = 0.9003891050583627 (τ = 0.0003891050583657588, cycle = 2314)
output at time t = 1.0003891050583624 (τ = 0.0003891050583657588, cycle = 2571)
output at time t = 1.100389105058362 (τ = 0.0003891050583657588, cycle = 2828)
output at time t = 1.2003891050583617 (τ = 0.0003891050583657588, cycle = 3085)
output at tim